In [2]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from transformers import AutoModelForCausalLM, AutoTokenizer
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
import io
import fitz
from googleapiclient.http import MediaIoBaseDownload
flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
import torch
creds = flow.run_local_server(port=0, open_browser=False)
service = build('drive', 'v3', credentials=creds)
print("Google Drive connected")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=16283401181-su0m8lu21oeg09ok91sur15jjfp3l779.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A33367%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=GDRi94tqOBzZI6b8j75pQSAhaHEUoF&code_challenge=214KP2zv4gf4wGEqeDXYCLq9WRgqoUqAMz6K86DOFoA&code_challenge_method=S256&access_type=offline
Google Drive connected


In [4]:
folder_name = 'gitlab-docs'

results = service.files().list(
    q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder'",
    fields='files(id, name)'
).execute()

folder = results.get('files', [])[0]
folder_id = folder['id']
print(f'Folder ID: {folder_id}')

files = service.files().list(
    q=f"'{folder_id}' in parents",
    fields='files(id, name, mimeType)'
).execute().get('files', [])

for f in files:
    print(f['name'])

Folder ID: 1TTF1gt7wmQdA1VjePdkVcn_Wahy0UJVj
GitLab Values _ The GitLab Handbook.pdf
Security at GitLab _ The GitLab Handbook.pdf
General & Entity Specific Benefits & Information _ The GitLab Handbook.pdf
GitLab Onboarding _ The GitLab Handbook.pdf
GitLab Internal Acceptable Use Policy _ The GitLab Handbook.pdf
GitLab Offboarding _ The GitLab Handbook.pdf
Anti-Harassment Policy _ The GitLab Handbook.pdf
United States Leave of Absence Policies _ The GitLab Handbook.pdf
People Policies _ The GitLab Handbook.pdf


In [5]:


def download_and_read_pdf(file_id):
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    
    done = False
    while not done:
        _, done = downloader.next_chunk()
    
    buffer.seek(0)
    text = ''
    doc = fitz.open(stream=buffer.read(), filetype='pdf')
    for page in doc:
        page_text = page.get_text()
        if page_text.strip():
            text += page_text + '\n'
    return text

docs = []
for f in files:
    print(f"reading: {f['name']}")
    text = download_and_read_pdf(f['id'])
    if text.strip():
        docs.append({'name': f['name'], 'text': text})
        print(f'{len(text)} chars')
    else:
        print('failed')

print(f'\nTotal: {len(docs)} files')

reading: GitLab Values _ The GitLab Handbook.pdf
120466 chars
reading: Security at GitLab _ The GitLab Handbook.pdf
20003 chars
reading: General & Entity Specific Benefits & Information _ The GitLab Handbook.pdf
29735 chars
reading: GitLab Onboarding _ The GitLab Handbook.pdf
8831 chars
reading: GitLab Internal Acceptable Use Policy _ The GitLab Handbook.pdf
19572 chars
reading: GitLab Offboarding _ The GitLab Handbook.pdf
38328 chars
reading: Anti-Harassment Policy _ The GitLab Handbook.pdf
39405 chars
reading: United States Leave of Absence Policies _ The GitLab Handbook.pdf
40056 chars
reading: People Policies _ The GitLab Handbook.pdf
35826 chars

Total: 9 files


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = []

for doc in docs:
    splits = splitter.split_text(doc['text'])
    for split in splits:
        chunks.append({'text': split, 'source': doc['name']})

print(f'chunks: {len(chunks)}')

embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

client = chromadb.PersistentClient(path='/home/hisham/enterprise-rag/chroma_db')

try:
    client.delete_collection('enterprise_docs')
except:
    pass

collection = client.get_or_create_collection('enterprise_docs')

texts = [chunk['text'] for chunk in chunks]
sources = [chunk['source'] for chunk in chunks]
ids = [str(i) for i in range(len(chunks))]

embeddings = embedding_model.encode(
    ["passage: " + t for t in texts],
    show_progress_bar=True
).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"source": s} for s in sources],
    ids=ids
)

print(f'{collection.count()} chunks saved')

chunks: 844


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

844 chunks saved


In [7]:
device = 'cuda'
base_model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map='auto',
    torch_dtype=torch.float16,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    local_files_only=True
)

def generate_with_qwen(messages):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors='pt').to(device)
    output = model.generate(
        inputs.input_ids,
        max_new_tokens=512,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None
    )
    output = output[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(output, skip_special_tokens=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
chat_history = []

def chat(question):
    question_embedding = embedding_model.encode(
        ["query: " + question]
    ).tolist()
    
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )

    context = '\n\n'.join(results['documents'][0])
    sources = [m['source'] for m in results['metadatas'][0]]

    history_text = ''
    for h in chat_history[-3:]:
        history_text += f"User: {h['question']}\nAssistant: {h['answer']}\n\n"

    messages = [
        {
            'role': 'system',
            'content': '\n'.join([
                'You are an enterprise HR assistant for GitLab.',
                'Answer questions based only on the provided context.',
                'If the answer is not in the context, say: I do not have information about that, please contact HR.',
                'Be concise and professional.'
            ])
        },
        {
            'role': 'user',
            'content': f'Previous conversation:\n{history_text}\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'
        }
    ]

    response = generate_with_qwen(messages)
    chat_history.append({'question': question, 'answer': response})

    print(f'\nAnswer: {response}')
    print(f'\nSources:')
    for source in set(sources):
        print(f'  - {source}')

In [14]:
chat_history = []

chat("What are GitLab's core values?")
chat("What is the acceptable use policy?")
chat("How does GitLab handle harassment complaints?")


Answer: GitLab's core values are Collaboration, Results for Customers, Efficiency, Diversity, Inclusion & Belonging, Iteration, and Transparency.

Sources:
  - GitLab Values _ The GitLab Handbook.pdf

Answer: The GitLab Internal Acceptable Use Policy can be found at this link: https://handbook.gitlab.com/handbook/people-group/acceptable-use-policy/. This document outlines guidelines for using GitLab resources to ensure a productive and safe environment for all users.

Sources:
  - GitLab Internal Acceptable Use Policy _ The GitLab Handbook.pdf

Answer: GitLab handles harassment complaints by allowing team members to bring their concerns to their People Business Partner. Upon receiving a complaint, the partner engages internal consultation services and responds accordingly. If the complaint is upheld, the alleged harasser may face disciplinary action up to dismissal for gross misconduct. If the complaint is not upheld, both parties involved may be supported in rebuilding their working 

In [16]:
collection = client.get_or_create_collection('enterprise_docs')
print(f"chunks: {collection.count()}")

# جرب بحث مباشر
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

q = "What are GitLab core values?"
q_emb = embedding_model.encode(["query: " + q]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

for doc in results['documents'][0]:
    print(doc[:200])
    print("---")

chunks: 844


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GitLab Values
Learn more about how we live our values at GitLab
CREDIT
GitLab’s six core values are 
 Collaboration, 
 Results for Customers, 
 Efficiency, 
Diversity, Inclusion & Belonging, 
 Iterati
---
co-founder, Sid Sijbrandij, has shared the origin of each of the CREDIT values, but just like the
rest of our work, we continually adjust our values and strive to make them better. GitLab
values are a
---
e group and other leaders accountable for upholding this
value.
Why have values
Our values provide guidelines on how to behave and are written to be actionable. They help
us describe the type of behav
---
